**Rebuild the duplicate-groups data from scratch in this fresh notebook, reusing Phase 1's logic**

In [1]:
import pandas as pd
import numpy as np
from rdkit import Chem

col_names = [
    "SMI", "lambda1_sTDA_nm", "F1_sTDA", "lambda1_TDDFT_nm", "F1_TDDFT",
    "lambda_max_exp_nm", "extinction", "solvent"
]
df_raw = pd.read_csv("paper_allDB.csv", header=None, skiprows=1, names=col_names)

df_exp = df_raw[df_raw["lambda_max_exp_nm"].notna()].copy()
df_range = df_exp[(df_exp["lambda_max_exp_nm"] >= 200) & (df_exp["lambda_max_exp_nm"] <= 900)].copy()
df_no254 = df_range[df_range["lambda_max_exp_nm"] != 254.0].copy()

df_no254["mol"] = df_no254["SMI"].apply(Chem.MolFromSmiles)
df_valid = df_no254[df_no254["mol"].notna()].copy()
df_valid["canonical_smi"] = df_valid["mol"].apply(Chem.MolToSmiles)

print(f"Valid, filtered records: {df_valid.shape[0]}")

dup_mask = df_valid.duplicated(subset="canonical_smi", keep=False)
dup_groups = df_valid[dup_mask].groupby("canonical_smi")["lambda_max_exp_nm"].agg(["count", "mean", "std", "min", "max"])
dup_groups["range"] = dup_groups["max"] - dup_groups["min"]

trustworthy = dup_groups[dup_groups["range"] <= 100]
print(f"Trustworthy duplicate groups: {trustworthy.shape[0]}")
print(trustworthy[["count", "std", "range"]].describe())

[09:00:29] WARNING: not removing hydrogen atom without neighbors
[09:00:29] WARNING: not removing hydrogen atom without neighbors
[09:00:29] WARNING: not removing hydrogen atom without neighbors
[09:00:29] WARNING: not removing hydrogen atom without neighbors
[09:00:29] WARNING: not removing hydrogen atom without neighbors
[09:00:29] WARNING: not removing hydrogen atom without neighbors
[09:00:29] WARNING: not removing hydrogen atom without neighbors
[09:00:29] WARNING: not removing hydrogen atom without neighbors
[09:00:29] WARNING: not removing hydrogen atom without neighbors
[09:00:29] WARNING: not removing hydrogen atom without neighbors
[09:00:29] WARNING: not removing hydrogen atom without neighbors
[09:00:30] Explicit valence for atom # 15 C, 5, is greater than permitted


[09:00:30] SMILES Parse Error: ring closure 1 duplicates bond between atom 46 and atom 47 for input: 'C(CN1c2cc(ccc2C(=C2C(=O)N(c3c2ccc(c3)c2ccc(cc2)C=C(C(=O)O)C#N)CC(C)CC)C1=O)c1ccc2c(c1)C1C1N2c1ccc(cc1)C)CC'
[09:00:30] WARNING: not removing hydrogen atom without neighbors
[09:00:30] Explicit valence for atom # 10 C, 5, is greater than permitted
[09:00:30] Explicit valence for atom # 20 C, 5, is greater than permitted
[09:00:30] Explicit valence for atom # 12 C, 5, is greater than permitted
[09:00:30] Explicit valence for atom # 12 C, 5, is greater than permitted
[09:00:30] Explicit valence for atom # 8 C, 5, is greater than permitted
[09:00:30] Explicit valence for atom # 11 C, 5, is greater than permitted
[09:00:30] Explicit valence for atom # 30 C, 5, is greater than permitted
[09:00:30] Explicit valence for atom # 8 C, 5, is greater than permitted
[09:00:30] Explicit valence for atom # 20 C, 5, is greater than permitted
[09:00:30] WARNING: not removing hydrogen atom without neighb

Valid, filtered records: 7257
Trustworthy duplicate groups: 54
           count        std      range
count  54.000000  54.000000  54.000000
mean    2.555556   6.793646  10.411111
std     1.487897   9.126087  13.429700
min     2.000000   0.000000   0.000000
25%     2.000000   1.103553   2.000000
50%     2.000000   4.242641   6.500000
75%     2.750000   9.192388  13.000000
max    10.000000  43.840620  62.000000


In [2]:
# Pooled standard deviation across all trustworthy duplicate groups
# Correct approach: sum of squared deviations from each group, divided by total degrees of freedom

pooled_ss = 0  # sum of squared deviations
pooled_df = 0  # degrees of freedom

for smi, group in df_valid[df_valid["canonical_smi"].isin(trustworthy.index)].groupby("canonical_smi"):
    vals = group["lambda_max_exp_nm"].values
    n = len(vals)
    if n > 1:
        group_mean = vals.mean()
        ss = np.sum((vals - group_mean) ** 2)
        pooled_ss += ss
        pooled_df += (n - 1)

pooled_std = np.sqrt(pooled_ss / pooled_df)
print(f"Pooled standard deviation (noise floor estimate): {pooled_std:.2f} nm")
print(f"Total degrees of freedom: {pooled_df}")
print(f"Number of duplicate groups contributing: {trustworthy.shape[0]}")

Pooled standard deviation (noise floor estimate): 10.07 nm
Total degrees of freedom: 84
Number of duplicate groups contributing: 54
